# KG1 Single-Model Analysis: llama-3.1-8b

**Two-stage approach:**
1. **Pilot** (Cell 3): 4 vignettes × 1 run — review before committing
2. **Full run** (Cell 5): 88 vignettes × 30 runs (~$1.33, ~45 min)

**Checkpoint/resume is automatic.** Re-running any cell resumes from where it left off.
All checkpoint files in `results/` are scanned — no data is lost even if Colab disconnects.

3. **Base analysis** (Cell 7): Aggregate metrics, Fisher+JSD edge detection, divergence taxonomy
4. **Enhanced KG2** (Cell 8–10): Bootstrap CIs, omnibus tests, direction, spurious edges, graph comparison

In [ ]:
import os, sys, json, shutil, glob
from pathlib import Path

from google.colab import drive, userdata
drive.mount('/content/drive')

PROJECT = Path('/content/drive/MyDrive/KG1_study')
PROJECT.mkdir(parents=True, exist_ok=True)
WORK = Path('/content/kg1_work')
WORK.mkdir(exist_ok=True)

required_files = [
    'vignette_battery.json',
    'llm_query_runner.py',
    'response_parser.py',
]
for f in required_files:
    src = PROJECT / f
    if src.exists():
        shutil.copy2(src, WORK / f)
        print(f'  + {f}')
    else:
        print(f'  MISSING: {f} — upload to Google Drive /KG1_study/')

sys.path.insert(0, str(WORK))

os.environ['TOGETHER_API_KEY'] = userdata.get('TOGETHER_API_KEY')
print('  + API key loaded')

import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'scipy'])
print('  + scipy installed')

RESULTS_DIR = str(PROJECT / 'results')
ANALYSIS_DIR = str(PROJECT / 'analysis')
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)
Path(ANALYSIS_DIR).mkdir(parents=True, exist_ok=True)
BATTERY = str(WORK / 'vignette_battery.json')

existing = sorted(glob.glob(f'{RESULTS_DIR}/run_*.jsonl'))
if existing:
    print(f'\nExisting checkpoints:')
    for f in existing:
        lines = sum(1 for l in open(f) if l.strip())
        print(f'  {Path(f).name}: {lines} records')
else:
    print('\nNo existing checkpoints — starting fresh.')

In [ ]:
from llm_query_runner import load_battery, MODEL_REGISTRY

items = load_battery(BATTERY)
baselines = [i for i in items if i['type'] == 'baseline']
perts = [i for i in items if i['type'] != 'baseline']
print(f'Battery: {len(items)} items ({len(baselines)} baselines, {len(perts)} perturbations)')
print(f'Families: {len(set(i["family"] for i in items))}')

cfg = MODEL_REGISTRY['llama-3.1-8b']
print(f'\nModel: llama-3.1-8b')
print(f'  API ID: {cfg["model_id"]}')
print(f'  Temperature: {cfg["temperature"]}')
print(f'  Max tokens: {cfg["max_tokens"]}')
print(f'  Price: ${cfg["price_in"]}/M in, ${cfg["price_out"]}/M out')

## Stage 1: Pilot Run
4 vignettes (1 baseline + 2 perturbations + 1 null), 1 run each.
Review the raw responses and parsed stances before launching the full battery.

In [ ]:
from llm_query_runner import (
    load_battery, load_completed, make_hash, run_single_query,
    RateLimiter, MODEL_REGISTRY
)
import time

PILOT_ITEMS = ['A1-BASE', 'A1-P1', 'A1-P2', 'A1-NULL1']
items = [i for i in load_battery(BATTERY) if i['id'] in PILOT_ITEMS]
mn = 'llama-3.1-8b'
n_runs = 1
api_key = os.environ['TOGETHER_API_KEY']

print(f'Pilot: {len(items)} items x {n_runs} run x 2 phases = {len(items)*n_runs*2} API calls')
print('=' * 60)

# Find or create checkpoint
results_path = Path(RESULTS_DIR)
existing_cp = sorted(results_path.glob('run_*.jsonl'))
if existing_cp:
    pilot_checkpoint = str(existing_cp[-1])
    print(f'Using existing checkpoint: {Path(pilot_checkpoint).name}')
else:
    from datetime import datetime, timezone
    pilot_checkpoint = str(results_path / f'run_{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")}.jsonl')
    print(f'New checkpoint: {Path(pilot_checkpoint).name}')

# Scan all checkpoints for completed work
done = set()
for jf in results_path.glob('run_*.jsonl'):
    done |= load_completed(str(jf))

work = [(i, mn, r) for i in items for r in range(n_runs)
        if make_hash(i['id'], mn, r) not in done]
print(f'Already done: {len(done)} | Remaining: {len(work)}')

if work:
    rl = RateLimiter()
    t0 = time.time()
    with open(pilot_checkpoint, 'a') as fout:
        for idx, (item, m, ri) in enumerate(work):
            print(f'[{idx+1}/{len(work)}] {item["id"]} r{ri}', end='')
            res = run_single_query(item, m, ri, rl, api_key, False)
            fout.write(json.dumps(res, default=str) + '\n')
            fout.flush()
            if res.get('error'):
                print(f' X {res["error"][:60]}')
            else:
                print(f' ok ({len(res.get("phase1_response","") or "")}+{len(res.get("phase2_response","") or "")}c)')
    print(f'\nDone in {(time.time()-t0)/60:.1f}m')
else:
    print('All pilot queries already completed!')

print(f'Checkpoint: {pilot_checkpoint}')

In [ ]:
from response_parser import extract_stances, detect_conditionality, detect_uncertainty

print('=' * 60)
print('PILOT RESPONSES')
print('=' * 60)

# Read from ALL checkpoint files to find pilot items
pilot_results = []
for jf in sorted(Path(RESULTS_DIR).glob('run_*.jsonl')):
    with open(jf) as f:
        for line in f:
            if not line.strip(): continue
            try:
                r = json.loads(line)
                if r.get('item_id') in PILOT_ITEMS and not r.get('error'):
                    pilot_results.append(r)
            except: pass

for r in pilot_results:
    print(f'\n{"=" * 60}')
    print(f'Item: {r["item_id"]} | Run: {r.get("run_idx", "?")}')
    print(f'{"=" * 60}')

    p1 = r.get('phase1_response', '') or ''
    p2 = r.get('phase2_response', '') or ''

    print(f'\n--- Phase 1 (first 600 chars) ---')
    print(p1[:600])
    if len(p1) > 600: print(f'  [...{len(p1)} chars total]')

    print(f'\n--- Phase 2 (first 800 chars) ---')
    print(p2[:800])
    if len(p2) > 800: print(f'  [...{len(p2)} chars total]')

    s1 = extract_stances(p1, 'phase1')
    s2 = extract_stances(p2, 'phase2')
    print(f'\n--- Extracted Stances ---')
    print(f'Phase 1: {len(s1)} treatments')
    for s in s1:
        print(f'  {s["treatment"]:25s} -> {s["stance"]:15s} (conf={s["confidence"]:.2f})')
    print(f'Phase 2: {len(s2)} treatments')
    for s in s2:
        print(f'  {s["treatment"]:25s} -> {s["stance"]:15s} (conf={s["confidence"]:.2f})')

    cond = detect_conditionality(p1 + '\n' + p2)
    unc = detect_uncertainty(p1 + '\n' + p2)
    print(f'\nConditionality: {cond} | Uncertainty: {unc}')

## Stage 2: Full Battery Run
88 items × 30 runs × 2 phases = 5,280 API calls. ~$1.33, ~45 min.

**Safe to interrupt and resume.** Re-running this cell automatically picks up where
you left off — it scans ALL existing checkpoint files for completed work.

In [ ]:
from llm_query_runner import (
    load_battery, load_completed, make_hash, run_single_query,
    RateLimiter, MODEL_REGISTRY
)
import time
from datetime import datetime, timezone

items = load_battery(BATTERY)
mn = 'llama-3.1-8b'
n_runs = 30
api_key = os.environ['TOGETHER_API_KEY']

print(f'Full battery: {len(items)} items x {n_runs} runs x 2 phases = {len(items)*n_runs*2} API calls')
print('=' * 60)

# Find or create checkpoint
results_path = Path(RESULTS_DIR)
existing_cp = sorted(results_path.glob('run_*.jsonl'))
if existing_cp:
    checkpoint = str(existing_cp[-1])
    print(f'Appending to: {Path(checkpoint).name}')
else:
    checkpoint = str(results_path / f'run_{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")}.jsonl')
    print(f'New checkpoint: {Path(checkpoint).name}')

# Scan ALL checkpoints for completed work
done = set()
for jf in results_path.glob('run_*.jsonl'):
    done |= load_completed(str(jf))

work = [(i, mn, r) for i in items for r in range(n_runs)
        if make_hash(i['id'], mn, r) not in done]
print(f'Already done: {len(done)} | Remaining: {len(work)}')

if work:
    rl = RateLimiter()
    n_done = n_err = 0
    t0 = time.time()
    with open(checkpoint, 'a') as fout:
        for item, m, ri in work:
            n_done += 1
            elapsed = time.time() - t0
            eta = (len(work) - n_done) / (n_done / elapsed) / 60 if n_done > 0 and elapsed > 0 else 0
            print(f'[{n_done}/{len(work)}] {item["id"]} x {m} r{ri} | ETA {eta:.0f}m', end='')

            res = None
            for att in range(3):
                res = run_single_query(item, m, ri, rl, api_key, False)
                if res['error'] is None:
                    break
                w = 2**att * 5
                print(f' [retry {att+1} in {w}s]', end='')
                time.sleep(w)

            fout.write(json.dumps(res, default=str) + '\n')
            fout.flush()
            if res.get('error'):
                n_err += 1
                print(f' X {res["error"][:60]}')
            else:
                print(f' ok ({len(res.get("phase1_response","") or "")}+{len(res.get("phase2_response","") or "")}c)')

    print(f'\nDone: {n_done} in {(time.time()-t0)/60:.1f}m, {n_err} errors')
else:
    print('All queries already completed!')

print(f'Checkpoint: {checkpoint}')

## Stage 3: Analysis
Runs base analysis (aggregate metrics, Fisher+JSD, divergence taxonomy).
Can run on partial results.

In [ ]:
from response_parser import run_analysis, parse_result, detect_edges
from collections import Counter

# Merge all checkpoint files (deduplicate by hash)
results_dir = Path(RESULTS_DIR)
all_jsonl = sorted(results_dir.glob('run_*.jsonl'))
if not all_jsonl:
    raise FileNotFoundError('No results files found — run Stage 2 first.')

seen_hashes = set()
all_results = []
for jf in all_jsonl:
    with open(jf) as f:
        for line in f:
            if not line.strip(): continue
            try:
                r = json.loads(line)
                h = r.get('hash', '')
                if h and h not in seen_hashes and r.get('error') is None:
                    seen_hashes.add(h)
                    all_results.append(r)
            except: pass

print(f'Loaded {len(all_results)} unique successful results from {len(all_jsonl)} file(s)')

merged_path = str(results_dir / 'merged_results.jsonl')
with open(merged_path, 'w') as f:
    for r in all_results:
        f.write(json.dumps(r, default=str) + '\n')

print('\n' + '=' * 60)
print('BASE ANALYSIS')
print('=' * 60)
metrics, kg2_base, edge_tests, divergences = run_analysis(merged_path, BATTERY, ANALYSIS_DIR)

print('\n' + '=' * 60)
print('RESULTS SUMMARY')
print('=' * 60)
for mn, m in sorted(metrics.items()):
    print(f'\n  {mn}:')
    print(f'    Recommendation accuracy: {m["rec_accuracy"]:.1%}')
    print(f'    Exclusion accuracy:      {m["exc_accuracy"]:.1%}')
    print(f'    Recommendation precision: {m["rec_precision"]:.1%}')
    if m.get('cond_rate') is not None: print(f'    Conditionality rate:     {m["cond_rate"]:.1%}')
    if m.get('unc_rate') is not None:  print(f'    Uncertainty rate:         {m["unc_rate"]:.1%}')
    if m.get('null_spec') is not None: print(f'    Null specificity:         {m["null_spec"]:.1%}')

if edge_tests:
    sig = sum(1 for t in edge_tests if t.get('significant'))
    print(f'\n  Edge tests: {len(edge_tests)} total, {sig} significant (BH-FDR q=0.05)')
if divergences:
    by_type = Counter(d['divergence_type'] for d in divergences)
    print(f'\n  Divergences: {dict(by_type)}')

## Stage 4: Enhanced KG2 Analysis
Defines the full enhanced KG2 module inline. Includes BCa bootstrap CIs on JSD,
permutation chi-squared omnibus test, direction analysis, spurious/phantom edge
detection, and publication-ready graph comparison metrics.

In [ ]:
#!/usr/bin/env python3
"""
KG2 Enhanced Assembly & Graph Comparison Module
================================================
Add-on to response_parser.py. Runs AFTER the base analysis pipeline.

Enhancements over base assemble_kg2():
1. Directed edges with correct/wrong direction classification
2. Continuous edge weights (mean JSD) with BCa bootstrap CIs
3. Conditionality testing (edge active in correct contexts only?)
4. Spurious edge detection from null perturbations
5. Omnibus distribution shift test (permutation chi-squared) before per-treatment
6. Publication-ready graph comparison metrics (SHD, edge-F1, SID-like, calibration)

Usage:
    python kg2_enhanced.py --results results/run_XXX.jsonl \
        --battery vignette_battery.json --outdir analysis

    Or import and call:
        from kg2_enhanced import run_enhanced_analysis
        results = run_enhanced_analysis(edge_tests, all_parsed, battery_items, kg2_base)
"""

import json, math, argparse
from pathlib import Path
from collections import defaultdict, Counter
from dataclasses import dataclass, field, asdict
from typing import Optional
import numpy as np

# =====================================================================
# DATA STRUCTURES
# =====================================================================

@dataclass
class KG2Edge:
    """Rich edge representation for a single model's implicit graph."""
    edge_id: str                          # KG1 statement ID (e.g., "S7R")
    model: str
    detected: bool                        # Significant in >50% of target-treatment probes
    soft_detected: bool = False           # Omnibus significant OR detection_rate > 0.25
    detection_rate: float = 0.0           # Fraction of probes where significant
    n_probes: int = 0                     # Number of target-treatment probes

    # Direction
    direction_correct: Optional[bool] = None  # Did distribution shift match KG1 prediction?
    direction_rate: float = 0.0               # Fraction of probes with correct direction

    # Weight (continuous)
    mean_jsd: float = 0.0                 # Mean JSD across probes
    median_jsd: float = 0.0
    jsd_ci_lower: float = 0.0            # 95% BCa bootstrap CI
    jsd_ci_upper: float = 0.0
    jsd_values: list = field(default_factory=list)  # Raw JSD per probe

    # Conditionality
    conditionality_tested: bool = False   # Was this edge tested across multiple contexts?
    conditionality_correct: Optional[bool] = None  # Active only in correct contexts?
    active_in_contexts: list = field(default_factory=list)   # Families where edge fired
    expected_contexts: list = field(default_factory=list)     # Families where edge should fire

    # Omnibus test
    omnibus_significant: Optional[bool] = None  # Permutation chi-sq significant?
    omnibus_p: Optional[float] = None


@dataclass
class KG1Edge:
    """Reference edge from Ferrari et al."""
    edge_id: str
    consensus_weight: float = 0.0  # Delphi agreement % (0-1)
    edge_type: str = "unknown"     # absolute_ci, relative_ci, indication, conditional
    families: list = field(default_factory=list)  # Which clinical families this applies to


@dataclass
class GraphComparison:
    """Publication-ready comparison metrics between KG1 and KG2."""
    model: str
    # Edge detection (hard: >50% probes significant)
    true_positives: int = 0        # KG1 edge present, KG2 detected (hard)
    false_negatives: int = 0       # KG1 edge present, KG2 missed (hard)
    false_positives: int = 0       # KG1 no edge, KG2 detected (spurious)
    # Derived
    precision: float = 0.0
    recall: float = 0.0
    f1: float = 0.0
    shd: int = 0                   # Structural Hamming Distance

    # Soft detection (omnibus significant OR det_rate >= 0.25)
    soft_true_positives: int = 0
    soft_recall: float = 0.0
    soft_f1: float = 0.0

    # Direction-sensitive
    direction_errors: int = 0      # Detected but wrong direction
    direction_accuracy: float = 0.0

    # Weighted metrics (by KG1 consensus strength)
    weighted_recall: float = 0.0   # Recall weighted by consensus weight
    weighted_precision: float = 0.0

    # Calibration
    weight_correlation: float = 0.0  # Correlation between KG1 weight and KG2 JSD
    calibration_error: float = 0.0   # Mean abs difference between normalised weights

    # Null specificity
    null_jsd_mean: float = 0.0     # Mean JSD on null perturbations (noise floor)
    null_jsd_95: float = 0.0       # 95th percentile of null JSD


# =====================================================================
# BOOTSTRAP JSD WITH BCa CONFIDENCE INTERVALS
# =====================================================================

def _jsd_from_counts(counts_a, counts_b, all_treatments, smoothing=0.5):
    """Compute JSD between two treatment count vectors with Dirichlet smoothing."""
    n_tx = len(all_treatments)
    p = np.array([(counts_a.get(t, 0) + smoothing) for t in all_treatments])
    q = np.array([(counts_b.get(t, 0) + smoothing) for t in all_treatments])
    p = p / p.sum()
    q = q / q.sum()

    m = 0.5 * (p + q)
    # Avoid log(0) — m is always > 0 due to smoothing
    kl_pm = np.sum(p * np.log(p / m))
    kl_qm = np.sum(q * np.log(q / m))
    return float(0.5 * kl_pm + 0.5 * kl_qm)


def bootstrap_jsd(samples_a, samples_b, n_boot=2000, alpha=0.05, smoothing=0.5):
    """
    Compute JSD between two sets of categorical samples with BCa bootstrap CI.

    Parameters:
        samples_a: list of treatment labels from condition A (N=30 runs)
        samples_b: list of treatment labels from condition B (N=30 runs)
        n_boot: number of bootstrap resamples
        alpha: significance level for CI (0.05 = 95% CI)
        smoothing: Dirichlet pseudocount for zero-count handling

    Returns:
        (jsd_observed, ci_lower, ci_upper)
    """
    all_treatments = sorted(set(samples_a + samples_b))
    if not all_treatments:
        return 0.0, 0.0, 0.0

    arr_a = np.array(samples_a)
    arr_b = np.array(samples_b)
    na, nb = len(arr_a), len(arr_b)

    def _counts(arr):
        c = Counter(arr.tolist() if hasattr(arr, 'tolist') else arr)
        return c

    # Observed JSD
    jsd_obs = _jsd_from_counts(_counts(arr_a), _counts(arr_b), all_treatments, smoothing)

    # Bootstrap resamples
    rng = np.random.default_rng(42)
    boot_jsds = np.empty(n_boot)
    for b in range(n_boot):
        ba = arr_a[rng.integers(0, na, size=na)]
        bb = arr_b[rng.integers(0, nb, size=nb)]
        boot_jsds[b] = _jsd_from_counts(_counts(ba), _counts(bb), all_treatments, smoothing)

    # BCa correction
    # Bias correction factor z0
    prop_less = np.sum(boot_jsds < jsd_obs) / n_boot
    from scipy.stats import norm
    z0 = norm.ppf(max(min(prop_less, 0.9999), 0.0001))

    # Acceleration factor from jackknife
    jack_vals = np.empty(na)
    for i in range(na):
        jack_a = np.concatenate([arr_a[:i], arr_a[i+1:]])
        jack_vals[i] = _jsd_from_counts(_counts(jack_a), _counts(arr_b), all_treatments, smoothing)
    jack_mean = jack_vals.mean()
    num = np.sum((jack_mean - jack_vals) ** 3)
    den = 6.0 * (np.sum((jack_mean - jack_vals) ** 2) ** 1.5)
    a_hat = num / den if abs(den) > 1e-12 else 0.0

    # Adjusted percentiles
    z_alpha = norm.ppf(alpha / 2)
    z_1alpha = norm.ppf(1 - alpha / 2)

    def _bca_percentile(z):
        num = z0 + z
        adj = z0 + num / (1.0 - a_hat * num) if abs(1.0 - a_hat * num) > 1e-12 else z0 + num
        return norm.cdf(adj)

    p_lo = _bca_percentile(z_alpha)
    p_hi = _bca_percentile(z_1alpha)

    ci_lo = float(np.quantile(boot_jsds, max(min(p_lo, 0.9999), 0.0001)))
    ci_hi = float(np.quantile(boot_jsds, max(min(p_hi, 0.9999), 0.0001)))

    return float(jsd_obs), ci_lo, ci_hi


# =====================================================================
# OMNIBUS DISTRIBUTION SHIFT TEST
# =====================================================================

def permutation_chi2(table_a, table_b, all_treatments, n_perm=5000):
    """
    Permutation-based chi-squared test for overall distribution shift.
    Tests H0: treatment distributions are identical between conditions A and B.

    Parameters:
        table_a: dict {treatment: count} for condition A (aggregated over N runs)
        table_b: dict {treatment: count} for condition B
        all_treatments: list of all treatment labels
        n_perm: number of permutations

    Returns:
        (chi2_observed, p_value)
    """
    # Build contingency matrix: rows = treatments, cols = conditions
    obs_a = np.array([table_a.get(t, 0) for t in all_treatments], dtype=float)
    obs_b = np.array([table_b.get(t, 0) for t in all_treatments], dtype=float)

    # Drop treatments with zero total (no information)
    mask = (obs_a + obs_b) > 0
    if mask.sum() < 2:
        return 0.0, 1.0  # Not enough categories

    obs_a = obs_a[mask]
    obs_b = obs_b[mask]

    def _chi2(a, b):
        total = a + b
        n_a = a.sum()
        n_b = b.sum()
        n_total = n_a + n_b
        if n_total == 0:
            return 0.0
        exp_a = total * n_a / n_total
        exp_b = total * n_b / n_total
        # Avoid division by zero
        exp_a = np.maximum(exp_a, 1e-10)
        exp_b = np.maximum(exp_b, 1e-10)
        return float(np.sum((a - exp_a)**2 / exp_a + (b - exp_b)**2 / exp_b))

    chi2_obs = _chi2(obs_a, obs_b)

    # Permutation: pool all counts and randomly reassign to conditions
    pooled = obs_a + obs_b
    n_a_total = int(obs_a.sum())
    n_total = int(pooled.sum())

    rng = np.random.default_rng(42)
    count_ge = 0
    for _ in range(n_perm):
        # Generate random split: sample n_a_total items from pooled
        # Multinomial approximation
        perm_a = rng.multinomial(n_a_total, pooled / pooled.sum())
        perm_b = pooled - perm_a
        perm_b = np.maximum(perm_b, 0)  # Guard against rounding
        chi2_perm = _chi2(perm_a.astype(float), perm_b.astype(float))
        if chi2_perm >= chi2_obs:
            count_ge += 1

    p_value = (count_ge + 1) / (n_perm + 1)  # +1 for continuity
    return chi2_obs, p_value


# =====================================================================
# DIRECTION ANALYSIS
# =====================================================================

def assess_direction(edge_test, battery_item):
    """
    Determine if the distribution shift is in the correct direction.

    For a perturbation that should EXCLUDE a treatment:
        Correct = recommendation rate decreased (base_rec_rate > pert_rec_rate)
    For a perturbation that should RECOMMEND a treatment:
        Correct = recommendation rate increased or stayed high

    Returns: True (correct), False (wrong), None (not assessable)
    """
    tx = edge_test["treatment"]
    exp_rec = set(edge_test.get("exp_rec", []))
    exp_exc = set(edge_test.get("exp_exc", []))
    base_rate = edge_test.get("base_rec_rate", 0)
    pert_rate = edge_test.get("pert_rec_rate", 0)

    if tx in exp_exc:
        # Perturbation should reduce this treatment
        return pert_rate < base_rate
    elif tx in exp_rec:
        # Perturbation should maintain or increase this treatment
        # But we need the baseline expectation too — if it was already recommended
        # in baseline, a non-decrease is correct
        return pert_rate >= base_rate - 0.1  # Allow small noise margin
    return None


# =====================================================================
# ENHANCED KG2 ASSEMBLY
# =====================================================================

def assemble_kg2_enhanced(edge_tests, all_parsed, battery_items):
    """
    Build enriched KG2 per model with direction, JSD weights, CIs, and omnibus tests.

    Parameters:
        edge_tests: list of dicts from detect_edges() in response_parser.py
        all_parsed: list of parsed results from parse_result()
        battery_items: dict {item_id: item_dict} from vignette battery

    Returns:
        dict {model_name: {edge_id: KG2Edge}}
    """
    # Group parsed results by (item_id, model)
    grouped = defaultdict(list)
    for p in all_parsed:
        grouped[(p["item_id"], p["model_name"])].append(p)

    # Group edge tests by (model, edge), but ONLY keep tests for target treatments
    # (those in exp_rec or exp_exc). Bystander treatments dilute edge detection rates.
    model_edge_tests = defaultdict(lambda: defaultdict(list))
    for t in edge_tests:
        tx = t["treatment"]
        target_tx = set(t.get("exp_rec", [])) | set(t.get("exp_exc", []))
        if tx not in target_tx:
            continue  # Skip bystander treatments — they don't test this edge
        for edge in t.get("edges", []):
            model_edge_tests[t["model"]][edge].append(t)

    # Also track ALL tests (including bystanders) for omnibus/spurious analysis
    model_edge_all = defaultdict(lambda: defaultdict(list))
    for t in edge_tests:
        for edge in t.get("edges", []):
            model_edge_all[t["model"]][edge].append(t)

    kg2 = {}
    for model_name, edges_dict in model_edge_tests.items():
        kg2[model_name] = {}

        for edge_id, tests in edges_dict.items():
            n_probes = len(tests)
            n_detected = sum(1 for t in tests if t.get("significant", False))

            # Direction analysis
            directions = []
            for t in tests:
                pert_item = battery_items.get(t["pert_id"])
                if pert_item:
                    d = assess_direction(t, pert_item)
                    if d is not None:
                        directions.append(d)

            n_correct_dir = sum(directions) if directions else 0
            direction_rate = n_correct_dir / len(directions) if directions else 0.0

            # JSD values
            jsd_values = [t["jsd"] for t in tests if t.get("jsd") is not None and t["jsd"] > 0]

            # Bootstrap CIs for mean JSD
            mean_jsd = float(np.mean(jsd_values)) if jsd_values else 0.0
            median_jsd = float(np.median(jsd_values)) if jsd_values else 0.0
            ci_lo, ci_hi = 0.0, 0.0

            if len(jsd_values) >= 3:
                # Simple bootstrap on JSD values (not on raw samples — we're aggregating)
                rng = np.random.default_rng(42)
                boot_means = []
                arr = np.array(jsd_values)
                for _ in range(2000):
                    boot_means.append(float(rng.choice(arr, size=len(arr), replace=True).mean()))
                ci_lo = float(np.percentile(boot_means, 2.5))
                ci_hi = float(np.percentile(boot_means, 97.5))

            # Omnibus test for the first perturbation (representative)
            # Use all tests (including bystanders) for a full-distribution omnibus
            omnibus_sig = None
            omnibus_p = None
            all_tests_for_edge = model_edge_all.get(model_name, {}).get(edge_id, tests)
            if all_tests_for_edge:
                t0 = all_tests_for_edge[0]
                base_parsed = grouped.get((t0["base_id"], model_name), [])
                pert_parsed = grouped.get((t0["pert_id"], model_name), [])
                if base_parsed and pert_parsed:
                    # Build stance count tables
                    base_counts = Counter()
                    pert_counts = Counter()
                    for p in base_parsed:
                        for s in p["stances"]:
                            base_counts[s["treatment"] + ":" + s["stance"]] += 1
                    for p in pert_parsed:
                        for s in p["stances"]:
                            pert_counts[s["treatment"] + ":" + s["stance"]] += 1

                    all_keys = sorted(set(list(base_counts.keys()) + list(pert_counts.keys())))
                    if len(all_keys) >= 2:
                        _, omnibus_p = permutation_chi2(
                            dict(base_counts), dict(pert_counts), all_keys, n_perm=2000
                        )
                        omnibus_sig = omnibus_p < 0.05

            # Conditionality: check if edge was tested across multiple families
            families_tested = list(set(t.get("pert_id", "")[:2] for t in tests))
            # (rough proxy — A=glottic_cT2, B=glottic_cT3, etc.)

            hard_detected = n_detected > n_probes / 2
            det_rate = n_detected / n_probes if n_probes > 0 else 0.0
            # Soft detection: omnibus significant OR at least 25% of probes significant
            soft_det = bool(omnibus_sig) or det_rate >= 0.25

            edge = KG2Edge(
                edge_id=edge_id,
                model=model_name,
                detected=hard_detected,
                soft_detected=soft_det,
                detection_rate=det_rate,
                n_probes=n_probes,
                direction_correct=direction_rate > 0.5 if directions else None,
                direction_rate=direction_rate,
                mean_jsd=mean_jsd,
                median_jsd=median_jsd,
                jsd_ci_lower=ci_lo,
                jsd_ci_upper=ci_hi,
                jsd_values=jsd_values,
                conditionality_tested=len(set(t.get("pert_id", "")[:1] for t in tests)) > 1,
                omnibus_significant=omnibus_sig,
                omnibus_p=omnibus_p,
            )
            kg2[model_name][edge_id] = edge

    return kg2


# =====================================================================
# SPURIOUS EDGE DETECTION
# =====================================================================

def detect_spurious_edges(all_parsed, battery_items, edge_tests):
    """
    Detect spurious edges: model responds to null perturbations (where it shouldn't).

    For each null perturbation, compute JSD between baseline and null.
    Any JSD above the noise threshold suggests surface-level sensitivity.

    Also checks: for treatments NOT in expected_recommendations or expected_excluded,
    did the model show significant shifts? These would be phantom edges.

    Returns:
        dict with:
            - null_jsd_by_model: {model: [jsd values on null perturbations]}
            - noise_threshold_by_model: {model: 95th percentile of null JSD}
            - phantom_edges: list of {model, pert_id, treatment, jsd, detail}
    """
    # Null perturbation JSDs
    null_jsd = defaultdict(list)
    for t in edge_tests:
        if t.get("type") == "null":
            null_jsd[t["model"]].append(t.get("jsd", 0))

    noise_thresholds = {}
    for model, jsds in null_jsd.items():
        if jsds:
            noise_thresholds[model] = float(np.percentile(jsds, 95))

    # Phantom edges: significant tests on treatments outside expectations
    phantom_edges = []
    for t in edge_tests:
        if not t.get("significant"):
            continue
        tx = t["treatment"]
        exp_rec = set(t.get("exp_rec", []))
        exp_exc = set(t.get("exp_exc", []))

        if tx not in exp_rec and tx not in exp_exc:
            # This treatment wasn't expected to be affected — spurious edge
            phantom_edges.append({
                "model": t["model"],
                "pert_id": t["pert_id"],
                "base_id": t["base_id"],
                "treatment": tx,
                "jsd": t.get("jsd", 0),
                "p_value": t.get("adjusted_p", t.get("p_value")),
                "detail": f"Significant shift in {tx} which is not in KG1 expectations for {t['pert_id']}"
            })

    return {
        "null_jsd_by_model": dict(null_jsd),
        "noise_thresholds": noise_thresholds,
        "phantom_edges": phantom_edges,
    }


# =====================================================================
# GRAPH COMPARISON METRICS
# =====================================================================

def compute_graph_comparison(kg2_enhanced, kg1_edges=None, spurious_data=None):
    """
    Compute publication-ready comparison metrics between KG1 and KG2.

    Parameters:
        kg2_enhanced: dict {model: {edge_id: KG2Edge}} from assemble_kg2_enhanced()
        kg1_edges: optional dict {edge_id: KG1Edge} with consensus weights
                   If None, all edges in KG2 are assumed to be in KG1 (confirmatory mode)
        spurious_data: output from detect_spurious_edges()

    Returns:
        dict {model: GraphComparison}
    """
    comparisons = {}

    for model, edges in kg2_enhanced.items():
        comp = GraphComparison(model=model)

        # All KG1 edges that were tested for this model
        kg1_tested = set(edges.keys())

        detected_edges = {eid for eid, e in edges.items() if e.detected}
        soft_detected_edges = {eid for eid, e in edges.items() if e.soft_detected}
        missed_edges = kg1_tested - detected_edges

        comp.true_positives = len(detected_edges)
        comp.false_negatives = len(missed_edges)
        comp.soft_true_positives = len(soft_detected_edges)

        # Spurious edges (false positives)
        if spurious_data:
            phantom = [p for p in spurious_data.get("phantom_edges", []) if p["model"] == model]
            comp.false_positives = len(set((p["pert_id"], p["treatment"]) for p in phantom))

        # Precision, recall, F1
        tp, fn, fp = comp.true_positives, comp.false_negatives, comp.false_positives
        comp.precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        comp.recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        comp.f1 = (2 * comp.precision * comp.recall / (comp.precision + comp.recall)
                    if (comp.precision + comp.recall) > 0 else 0.0)

        # Soft detection metrics
        stp = comp.soft_true_positives
        sfn = len(kg1_tested) - stp
        comp.soft_recall = stp / (stp + sfn) if (stp + sfn) > 0 else 0.0
        soft_prec = stp / (stp + fp) if (stp + fp) > 0 else 0.0
        comp.soft_f1 = (2 * soft_prec * comp.soft_recall / (soft_prec + comp.soft_recall)
                        if (soft_prec + comp.soft_recall) > 0 else 0.0)

        # SHD = false positives + false negatives + direction errors
        direction_errors = sum(
            1 for e in edges.values()
            if e.detected and e.direction_correct is False
        )
        comp.direction_errors = direction_errors
        comp.shd = fp + len(missed_edges) + direction_errors

        # Direction accuracy (among detected edges)
        detected_with_dir = [e for e in edges.values() if e.detected and e.direction_correct is not None]
        if detected_with_dir:
            comp.direction_accuracy = sum(
                1 for e in detected_with_dir if e.direction_correct
            ) / len(detected_with_dir)

        # Weighted metrics (by KG1 consensus weight if available)
        if kg1_edges:
            w_tp = sum(kg1_edges[eid].consensus_weight for eid in detected_edges if eid in kg1_edges)
            w_all = sum(kg1_edges[eid].consensus_weight for eid in kg1_tested if eid in kg1_edges)
            comp.weighted_recall = w_tp / w_all if w_all > 0 else 0.0

            # Calibration: correlation between KG1 consensus weight and KG2 JSD
            weights_kg1 = []
            weights_kg2 = []
            for eid, e in edges.items():
                if eid in kg1_edges and e.mean_jsd > 0:
                    weights_kg1.append(kg1_edges[eid].consensus_weight)
                    weights_kg2.append(e.mean_jsd)
            if len(weights_kg1) >= 3:
                from scipy.stats import spearmanr
                corr, _ = spearmanr(weights_kg1, weights_kg2)
                comp.weight_correlation = float(corr) if not np.isnan(corr) else 0.0

                # Calibration error: normalise both to [0,1] and compute MAE
                kg1_norm = np.array(weights_kg1) / max(weights_kg1)
                kg2_norm = np.array(weights_kg2) / max(weights_kg2) if max(weights_kg2) > 0 else np.zeros_like(weights_kg2)
                comp.calibration_error = float(np.mean(np.abs(kg1_norm - kg2_norm)))

        # Null specificity
        if spurious_data:
            null_jsds = spurious_data.get("null_jsd_by_model", {}).get(model, [])
            if null_jsds:
                comp.null_jsd_mean = float(np.mean(null_jsds))
                comp.null_jsd_95 = float(np.percentile(null_jsds, 95))

        comparisons[model] = comp

    return comparisons


# =====================================================================
# ENHANCED REPORT
# =====================================================================

def generate_enhanced_report(kg2_enhanced, comparisons, spurious_data, out_path):
    """Generate publication-ready markdown report."""
    L = []
    L.append("# KG2 Enhanced Analysis Report\n")
    L.append("## 1. Graph Comparison Metrics\n")

    models = sorted(comparisons.keys())
    L.append("| Model | TP | FN | FP | Prec | Recall | F1 | Soft TP | Soft Recall | Soft F1 | SHD | Dir Acc | Null JSD 95 |")
    L.append("|---|---|---|---|---|---|---|---|---|---|---|---|---|")
    for m in models:
        c = comparisons[m]
        L.append(
            f"| {m} | {c.true_positives} | {c.false_negatives} | {c.false_positives} "
            f"| {c.precision:.2f} | {c.recall:.2f} | {c.f1:.2f} "
            f"| {c.soft_true_positives} | {c.soft_recall:.2f} | {c.soft_f1:.2f} "
            f"| {c.shd} | {c.direction_accuracy:.0%} | {c.null_jsd_95:.4f} |"
        )

    L.append("\n## 2. Edge-Level Detail\n")
    for m in models:
        L.append(f"\n### {m}\n")
        L.append("| Edge | Hard | Soft | Dir | JSD | 95% CI | Omnibus p | N |")
        L.append("|---|---|---|---|---|---|---|---|")
        for eid in sorted(kg2_enhanced[m].keys()):
            e = kg2_enhanced[m][eid]
            hard = "+" if e.detected else "-"
            soft = "+" if e.soft_detected else "-"
            dir_s = "OK" if e.direction_correct else ("WRONG" if e.direction_correct is False else "?")
            ci = f"[{e.jsd_ci_lower:.3f}, {e.jsd_ci_upper:.3f}]" if e.jsd_ci_lower > 0 else "-"
            omni = f"{e.omnibus_p:.3f}" if e.omnibus_p is not None else "-"
            L.append(f"| {eid} | {hard} ({e.detection_rate:.0%}) | {soft} | {dir_s} ({e.direction_rate:.0%}) "
                     f"| {e.mean_jsd:.4f} | {ci} | {omni} | {e.n_probes} |")

    L.append("\n## 3. Spurious Edges\n")
    phantoms = spurious_data.get("phantom_edges", [])
    if phantoms:
        L.append(f"Total phantom edges detected: {len(phantoms)}\n")
        by_model = defaultdict(list)
        for p in phantoms:
            by_model[p["model"]].append(p)
        for m in sorted(by_model.keys()):
            L.append(f"\n**{m}**: {len(by_model[m])} phantom edges")
            for p in by_model[m][:10]:
                L.append(f"- {p['pert_id']} x {p['treatment']}: JSD={p['jsd']:.4f}, p={p['p_value']:.4f}")
    else:
        L.append("No phantom edges detected.\n")

    L.append("\n## 4. Noise Floor (Null Perturbations)\n")
    for m in models:
        nulls = spurious_data.get("null_jsd_by_model", {}).get(m, [])
        if nulls:
            L.append(f"- **{m}**: mean={np.mean(nulls):.4f}, "
                     f"median={np.median(nulls):.4f}, "
                     f"95th={np.percentile(nulls, 95):.4f}, "
                     f"max={max(nulls):.4f}")

    report = "\n".join(L)
    with open(out_path, "w") as f:
        f.write(report)
    return report


# =====================================================================
# MAIN PIPELINE
# =====================================================================

def run_enhanced_analysis(edge_tests, all_parsed, battery_items, output_dir=None):
    """
    Run full enhanced KG2 analysis. Call after base response_parser.run_analysis().

    Parameters:
        edge_tests: from response_parser.detect_edges()
        all_parsed: from response_parser.parse_result() applied to all results
        battery_items: dict {item_id: item_dict}
        output_dir: path to save outputs (optional)

    Returns:
        (kg2_enhanced, comparisons, spurious_data)
    """
    print("=" * 60)
    print("KG2 ENHANCED ANALYSIS")
    print("=" * 60)

    # 1. Enhanced KG2 assembly
    print("\n[1/4] Assembling enhanced KG2...")
    kg2 = assemble_kg2_enhanced(edge_tests, all_parsed, battery_items)
    for m in sorted(kg2.keys()):
        n_hard = sum(1 for e in kg2[m].values() if e.detected)
        n_soft = sum(1 for e in kg2[m].values() if e.soft_detected)
        n_total = len(kg2[m])
        mean_jsd = np.mean([e.mean_jsd for e in kg2[m].values() if e.mean_jsd > 0]) if kg2[m] else 0
        print(f"  {m}: {n_hard}/{n_total} hard, {n_soft}/{n_total} soft detected, mean JSD={mean_jsd:.4f}")

    # 2. Spurious edge detection
    print("\n[2/4] Detecting spurious edges...")
    spurious_data = detect_spurious_edges(all_parsed, battery_items, edge_tests)
    n_phantom = len(spurious_data.get("phantom_edges", []))
    print(f"  Phantom edges: {n_phantom}")
    for m, jsds in spurious_data.get("null_jsd_by_model", {}).items():
        print(f"  {m} null JSD: mean={np.mean(jsds):.4f}, 95th={np.percentile(jsds, 95):.4f}")

    # 3. Graph comparison
    print("\n[3/4] Computing graph comparison metrics...")
    comparisons = compute_graph_comparison(kg2, kg1_edges=None, spurious_data=spurious_data)
    for m in sorted(comparisons.keys()):
        c = comparisons[m]
        print(f"  {m}: P={c.precision:.2f} R={c.recall:.2f} F1={c.f1:.2f} "
              f"SoftR={c.soft_recall:.2f} SoftF1={c.soft_f1:.2f} "
              f"SHD={c.shd} DirAcc={c.direction_accuracy:.0%}")

    # 4. Save outputs
    if output_dir:
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)

        # KG2 as serialisable dict
        kg2_serial = {}
        for m, edges in kg2.items():
            kg2_serial[m] = {}
            for eid, e in edges.items():
                d = asdict(e)
                d.pop("jsd_values", None)  # Don't dump raw arrays
                kg2_serial[m][eid] = d
        with open(out / "kg2_enhanced.json", "w") as f:
            json.dump(kg2_serial, f, indent=2)

        # Comparisons
        comp_serial = {m: asdict(c) for m, c in comparisons.items()}
        with open(out / "graph_comparison.json", "w") as f:
            json.dump(comp_serial, f, indent=2)

        # Spurious
        with open(out / "spurious_edges.json", "w") as f:
            json.dump(spurious_data, f, indent=2, default=str)

        # Report
        print("\n[4/4] Generating report...")
        generate_enhanced_report(kg2, comparisons, spurious_data, out / "kg2_report.md")
        print(f"  Saved to: {out}/")

    return kg2, comparisons, spurious_data


print('KG2 Enhanced module loaded.')

In [ ]:
all_parsed = [p for p in (parse_result(r) for r in all_results) if p]
print(f'Parsed {len(all_parsed)} responses for enhanced analysis')

with open(BATTERY) as f:
    bat = json.load(f)
battery_items = {}
for b in bat['baselines']:  battery_items[b['id']] = b
for p in bat['perturbations']: battery_items[p['id']] = p

kg2_enh, comparisons, spurious_data = run_enhanced_analysis(
    edge_tests, all_parsed, battery_items, output_dir=ANALYSIS_DIR
)
print(f'\nAll outputs saved to: {ANALYSIS_DIR}')

## Stage 5: Visualisation

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if 'metrics' in dir() and metrics:
    models_list = sorted(metrics.keys())

    # Plot 1: Aggregate Performance
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for idx, (key, label) in enumerate([
        ('rec_accuracy', 'Recommendation\nAccuracy'),
        ('exc_accuracy', 'Exclusion\nAccuracy'),
        ('rec_precision', 'Recommendation\nPrecision'),
    ]):
        ax = axes[idx]
        vals = [metrics[m][key] for m in models_list]
        bars = ax.bar(range(len(models_list)), vals, color='#4C72B0')
        ax.set_xticks(range(len(models_list)))
        ax.set_xticklabels(models_list, rotation=45, ha='right', fontsize=9)
        ax.set_ylabel(label); ax.set_ylim(0, 1.05)
        ax.axhline(1.0, color='gray', ls='--', alpha=0.3)
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width()/2, v + 0.02, f'{v:.0%}', ha='center', fontsize=10)
    fig.suptitle('llama-3.1-8b: Aggregate Performance', fontweight='bold')
    plt.tight_layout()
    plt.savefig(str(Path(ANALYSIS_DIR) / 'agg_perf.png'), dpi=150, bbox_inches='tight')
    plt.show()

    # Plot 2: Enhanced Edge Detail
    if 'kg2_enh' in dir() and kg2_enh:
        for model_name, edges in kg2_enh.items():
            all_e = sorted(edges.keys())
            if not all_e: continue
            det_rates = [edges[e].detection_rate for e in all_e]
            soft_flags = [edges[e].soft_detected for e in all_e]
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, max(8, len(all_e)*0.35)))
            colors = ['#2ecc71' if s else '#e74c3c' for s in soft_flags]
            y_pos = range(len(all_e))
            ax1.barh(y_pos, det_rates, color=colors, alpha=0.8)
            ax1.axvline(0.5, color='red', ls='--', alpha=0.5, label='Hard (50%)')
            ax1.axvline(0.25, color='orange', ls='--', alpha=0.5, label='Soft (25%)')
            ax1.set_yticks(y_pos); ax1.set_yticklabels(all_e, fontsize=8)
            ax1.set_xlabel('Detection Rate'); ax1.set_xlim(0, 1.05)
            ax1.set_title('Edge Detection Rate (green=soft detected)', fontweight='bold', fontsize=10)
            ax1.legend(fontsize=8)
            means = [edges[e].mean_jsd for e in all_e]
            ci_lo = [edges[e].jsd_ci_lower for e in all_e]
            ci_hi = [edges[e].jsd_ci_upper for e in all_e]
            err_lo = [max(0, m - lo) for m, lo in zip(means, ci_lo)]
            err_hi = [max(0, hi - m) for m, hi in zip(means, ci_hi)]
            ax2.barh(y_pos, means, color='#3498db', alpha=0.7)
            ax2.errorbar(means, y_pos, xerr=[err_lo, err_hi], fmt='none', ecolor='black', capsize=3)
            ax2.set_yticks(y_pos); ax2.set_yticklabels(all_e, fontsize=8)
            ax2.set_xlabel('Mean JSD (95% Bootstrap CI)')
            ax2.set_title('Edge Weight (JSD)', fontweight='bold', fontsize=10)
            plt.suptitle(model_name, fontweight='bold', fontsize=11)
            plt.tight_layout()
            plt.savefig(str(Path(ANALYSIS_DIR) / 'enhanced_edge_detail.png'), dpi=150, bbox_inches='tight')
            plt.show()

    # Plot 3: JSD Distribution
    if edge_tests:
        fig, ax = plt.subplots(figsize=(10, 5))
        jsds = [t['jsd'] for t in edge_tests if t.get('jsd', 0) > 0]
        if jsds:
            ax.hist(jsds, bins=30, color='#4C72B0', alpha=0.8, edgecolor='white')
            ax.axvline(np.median(jsds), color='red', ls='--', label=f'Median: {np.median(jsds):.3f}')
            if 'spurious_data' in dir() and spurious_data:
                for m, t in spurious_data.get('noise_thresholds', {}).items():
                    ax.axvline(t, color='orange', ls=':', label=f'Null 95th: {t:.3f}')
            ax.set_xlabel('JSD'); ax.set_ylabel('Count')
            ax.set_title('Edge Weight Distribution', fontweight='bold'); ax.legend()
        plt.tight_layout()
        plt.savefig(str(Path(ANALYSIS_DIR) / 'jsd_dist.png'), dpi=150, bbox_inches='tight')
        plt.show()

    # Plot 4: Direction accuracy
    if 'kg2_enh' in dir() and kg2_enh:
        for model_name, edges in kg2_enh.items():
            all_e = sorted(edges.keys())
            if not all_e: continue
            dir_rates = [edges[e].direction_rate for e in all_e]
            dir_colors = ['#2ecc71' if r > 0.5 else '#e74c3c' for r in dir_rates]
            fig, ax = plt.subplots(figsize=(10, max(6, len(all_e)*0.3)))
            ax.barh(range(len(all_e)), dir_rates, color=dir_colors, alpha=0.8)
            ax.axvline(0.5, color='gray', ls='--', alpha=0.5, label='Chance')
            ax.set_yticks(range(len(all_e))); ax.set_yticklabels(all_e, fontsize=8)
            ax.set_xlabel('Direction Accuracy'); ax.set_xlim(0, 1.05)
            ax.set_title(f'{model_name}: Direction Accuracy', fontweight='bold'); ax.legend()
            plt.tight_layout()
            plt.savefig(str(Path(ANALYSIS_DIR) / 'direction_accuracy.png'), dpi=150, bbox_inches='tight')
            plt.show()

    print(f'Plots saved to: {ANALYSIS_DIR}')
else:
    print('No metrics — run analysis cells first.')

In [ ]:
from collections import defaultdict, Counter

if 'divergences' in dir() and divergences:
    print('=' * 60)
    print('DIVERGENCE TAXONOMY')
    print('=' * 60)
    by_type = Counter(d['divergence_type'] for d in divergences)
    for dt, n in by_type.most_common():
        print(f'\n  {dt}: {n} instances')
        for d in [x for x in divergences if x['divergence_type'] == dt][:5]:
            print(f'    - {d["pert_id"]}: {d["detail"]}')

    if edge_tests:
        print('\n\nMost-missed edges:')
        missed = defaultdict(int); tested = defaultdict(int)
        for t in edge_tests:
            for e in t['edges']:
                tested[e] += 1
                if not t.get('significant', False): missed[e] += 1
        for e, n in sorted(missed.items(), key=lambda x: -x[1])[:15]:
            print(f'    {e}: missed {n}/{tested[e]} ({n/tested[e]:.0%})')

if 'comparisons' in dir() and comparisons:
    print('\n' + '=' * 60)
    print('ENHANCED GRAPH COMPARISON')
    print('=' * 60)
    for m, c in sorted(comparisons.items()):
        print(f'\n  {m}:')
        print(f'    Hard: TP={c.true_positives} FN={c.false_negatives} FP={c.false_positives}')
        print(f'    Hard:  P={c.precision:.2f} R={c.recall:.2f} F1={c.f1:.2f}')
        print(f'    Soft:  TP={c.soft_true_positives} R={c.soft_recall:.2f} F1={c.soft_f1:.2f}')
        print(f'    SHD={c.shd} | Direction accuracy={c.direction_accuracy:.0%}')
        if c.null_jsd_95 > 0:
            print(f'    Noise floor: mean={c.null_jsd_mean:.4f}, 95th={c.null_jsd_95:.4f}')

if 'spurious_data' in dir() and spurious_data:
    phantoms = spurious_data.get('phantom_edges', [])
    if phantoms:
        print(f'\n  Phantom edges: {len(phantoms)}')
        for p in phantoms[:10]:
            print(f'    - {p["pert_id"]} x {p["treatment"]}: JSD={p["jsd"]:.4f}')